In [1]:
import tushare as ts
import pandas as pd
from config.config import *

token = '387b85f9f3e3d3894239d5c7fb8c57cf4e47330c1c04513d08e2e5356887'
pro = ts.pro_api(token)
pro._DataApi__token = token
pro._DataApi__http_url = 'http://lianghua.nanyangqiankun.top'
NUM = 5

In [3]:
# df = pd.read_csv("data/sample_funds.csv")
df = pd.read_csv("data/old_funds.csv")
df

,ts_code,name,found_date,fund_type,invest_type,benchmark
0,001917.OF,招商量化精选A,20160315,股票型,股票型,中证500指数收益率*80%+中债综合指数收益率*20%
1,002229.OF,华夏经济转型,20160315,股票型,股票型,沪深300指数收益率*90%+上证国债指数收益率*10%
2,001975.OF,景顺长城环保优势,20160315,股票型,股票型,中证环保产业指数收益率*40%+沪深300指数收益率*40%+中证全债指数收益率*20%
3,002334.OF,汇丰晋信大盘波动精选A,20160311,股票型,股票型,沪深300指数*90%+同业存款利率(税后)*10%
4,001718.OF,工银物流产业A,20160301,股票型,股票型,沪深300运输指数收益率*80%+中债综合财富(总值)指数收益率*20%
...,...,...,...,...,...,...
176,160603.OF,鹏华普天收益,20030712,混合型,收益型,沪深300指数收益率*70%+中证综合债指数收益率*30%
177,070002.OF,嘉实增长,20030709,混合型,增长型,巨潮500(小盘)指数收益率*60%+中债总指数收益率*40%
178,070003.OF,嘉实稳健,20030709,混合型,稳健型,巨潮200(大盘)指数收益率*60%+中债总指数收益率*40%
179,090001.OF,大成价值增长A,20021111,混合型,价值型,沪深300指数*80%+中债综合指数*20%


In [40]:
count = 0
total = len(df["ts_code"])
df['ts_code']

0      002552.OF
1      002489.OF
2      002450.OF
3      002407.OF
4      002067.OF
         ...    
503    121001.OF
504    001001.OF
505    213001.OF
506    161601.OF
507    020001.OF
Name: ts_code, Length: 508, dtype: object

In [4]:
import time
from tqdm import tqdm
count = 0
ndf = pd.DataFrame()

for code in tqdm(df['ts_code'], desc="基金代码处理中"):
    count += 1
    # print(f"{count}/{total} : {code}")

    df_0 = pro.fund_nav(ts_code = code, fields=['ts_code', 'ann_date', 'nav_date', 'unit_navm', 'accum_nav', 'adj_nav'])
    # df_0['ann_date'] = pd.to_datetime(df_0['ann_date'], format="%Y%m%d")
    df_0.to_csv(f'data/funds/{code}.csv', index=False)
    ndf = pd.concat([ndf, df_0], ignore_index=True)

    # if count % 80 == 0:
    #     print(f"\n⏸ 已跑完 {count} 个，暂停 60 秒...\n")
    #     time.sleep(60)  # tushare权限不够，一分钟最多调用80个

count =0


基金代码处理中: 100%|██████████| 181/181 [01:17<00:00,  2.33it/s]


In [5]:
# import time
# count = 0
# ndf = pd.DataFrame()

# for code in df['ts_code']:
#     count += 1 
#     print(f"{count}/{total} : {code}")

#     df = pd.read_csv(f'data/funds/{code}.csv')
#     # df['ann_date'] = pd.to_datetime(df['ann_date'], format="%Y%m%d")
#     df0 = df[['ts_code','ann_date','adj_nav']].copy()
#     ndf = pd.concat([ndf, df0], ignore_index=True)

# count =0


In [6]:
ndf

,ts_code,ann_date,nav_date,accum_nav,adj_nav
0,001917.OF,20260331,20260330,3.6357,3.795320
1,001917.OF,20260328,20260327,3.6373,3.797023
2,001917.OF,20260327,20260326,3.5994,3.756683
3,001917.OF,20260326,20260325,3.6405,3.800429
4,001917.OF,20260325,20260324,3.5807,3.736779
...,...,...,...,...,...
627022,040001.OF,20010928,20010927,1.0000,1.000000
627023,040001.OF,20010927,20010926,1.0000,1.000000
627024,040001.OF,20010926,20010925,1.0000,1.000000
627025,040001.OF,20010925,20010924,1.0000,1.000000


In [7]:
ndf["ann_date"] = pd.to_datetime(ndf["ann_date"])
ndf.set_index(["ann_date", "ts_code"], inplace=True)
ndf.sort_index(inplace=True)
ndf

nav_date  accum_nav   adj_nav
ann_date   ts_code                                 
2001-09-22 040001.OF  20010921     1.0000  1.000000
2001-09-25 040001.OF  20010924     1.0000  1.000000
2001-09-26 040001.OF  20010925     1.0000  1.000000
2001-09-27 040001.OF  20010926     1.0000  1.000000
2001-09-28 040001.OF  20010927     1.0000  1.000000
...                        ...        ...       ...
2026-03-31 540008.OF  20260330     2.9249  3.079717
           540009.OF  20260330     1.7303  1.714082
           540010.OF  20260330     4.1115  4.111500
           550008.OF  20260330     2.9682  3.523751
           960000.OF  20260330     2.2753  2.275300

[627027 rows x 3 columns]

In [8]:
# 截断以下，只要1年内数据
import datetime  # 加在文件最上面

ndf = ndf.loc[ndf.index.get_level_values("ann_date") >= ten_year_ago]
ndf

nav_date  accum_nav    adj_nav
ann_date   ts_code                                  
2016-03-30 000011.OF  20160329    13.6370  16.694700
           000309.OF  20160329     1.4450   1.445000
           000409.OF  20160329     1.3490   1.349000
           000471.OF  20160329     2.2190   2.219000
           000513.OF  20160329     1.3610   1.361000
...                        ...        ...        ...
2026-03-31 540008.OF  20260330     2.9249   3.079717
           540009.OF  20260330     1.7303   1.714082
           540010.OF  20260330     4.1115   4.111500
           550008.OF  20260330     2.9682   3.523751
           960000.OF  20260330     2.2753   2.275300

[494930 rows x 3 columns]

In [9]:
ndf.to_feather('data/funds.feather')

In [10]:
ndf

nav_date  accum_nav    adj_nav
ann_date   ts_code                                  
2016-03-30 000011.OF  20160329    13.6370  16.694700
           000309.OF  20160329     1.4450   1.445000
           000409.OF  20160329     1.3490   1.349000
           000471.OF  20160329     2.2190   2.219000
           000513.OF  20160329     1.3610   1.361000
...                        ...        ...        ...
2026-03-31 540008.OF  20260330     2.9249   3.079717
           540009.OF  20260330     1.7303   1.714082
           540010.OF  20260330     4.1115   4.111500
           550008.OF  20260330     2.9682   3.523751
           960000.OF  20260330     2.2753   2.275300

[494930 rows x 3 columns]

In [12]:
print(len(ndf.loc[five_year_ago]))
print(len(ndf.loc['2021-04-01']))
print(len(ndf.loc['2025-03-28']))
print(len(ndf.loc['2025-12-05']))
print(len(ndf.loc['2026-03-26']))

183
181
362
362
181


#### 下面开爬基金规模

In [23]:
# df = pd.read_csv("data/sample_funds.csv")
df = pd.read_csv("data/old_funds.csv")
df

,ts_code,name,found_date,fund_type,invest_type,benchmark
0,001917.OF,招商量化精选A,20160315,股票型,股票型,中证500指数收益率*80%+中债综合指数收益率*20%
1,002229.OF,华夏经济转型,20160315,股票型,股票型,沪深300指数收益率*90%+上证国债指数收益率*10%
2,001975.OF,景顺长城环保优势,20160315,股票型,股票型,中证环保产业指数收益率*40%+沪深300指数收益率*40%+中证全债指数收益率*20%
3,002334.OF,汇丰晋信大盘波动精选A,20160311,股票型,股票型,沪深300指数*90%+同业存款利率(税后)*10%
4,001718.OF,工银物流产业A,20160301,股票型,股票型,沪深300运输指数收益率*80%+中债综合财富(总值)指数收益率*20%
...,...,...,...,...,...,...
176,160603.OF,鹏华普天收益,20030712,混合型,收益型,沪深300指数收益率*70%+中证综合债指数收益率*30%
177,070002.OF,嘉实增长,20030709,混合型,增长型,巨潮500(小盘)指数收益率*60%+中债总指数收益率*40%
178,070003.OF,嘉实稳健,20030709,混合型,稳健型,巨潮200(大盘)指数收益率*60%+中债总指数收益率*40%
179,090001.OF,大成价值增长A,20021111,混合型,价值型,沪深300指数*80%+中债综合指数*20%


In [ ]:
# 基金规模
import time
count = 0
sdf0 = pd.DataFrame()
total = len(df["ts_code"])
for code in tqdm(df['ts_code'], desc="爬基金规模数据中..."):
    count += 1
    print(f"{count}/{total} : {code}")

    df0 = pro.fund_share(ts_code = code, start_date = ten_year_ago)
    df0['trade_date'] = pd.to_datetime(df0['trade_date'], format="%Y%m%d")
    df0.to_csv(f'data/funds_share/{code}.csv', index=False)
    sdf0 = pd.concat([sdf0, df0], ignore_index=True)

count =0
sdf0


1/181 : 001917.OF
2/181 : 002229.OF
3/181 : 001975.OF
4/181 : 002334.OF
5/181 : 001718.OF
6/181 : 002168.OF
7/181 : 001645.OF
8/181 : 002300.OF
9/181 : 001717.OF
10/181 : 001719.OF
11/181 : 001223.OF
12/181 : 001956.OF
13/181 : 002210.OF
14/181 : 001877.OF
15/181 : 960000.OF
16/181 : 001714.OF
17/181 : 001616.OF
18/181 : 001849.OF
19/181 : 001637.OF
20/181 : 001915.OF
21/181 : 001726.OF
22/181 : 001677.OF
23/181 : 001705.OF
24/181 : 001938.OF
25/181 : 001736.OF
26/181 : 001766.OF
27/181 : 001643.OF
28/181 : 001626.OF
29/181 : 001672.OF
30/181 : 001692.OF
31/181 : 001528.OF
32/181 : 001628.OF
33/181 : 001541.OF
34/181 : 001583.OF
35/181 : 001482.OF
36/181 : 001651.OF
37/181 : 001542.OF
38/181 : 001410.OF
39/181 : 001473.OF
40/181 : 001577.OF
41/181 : 519965.OF
42/181 : 001490.OF
43/181 : 519714.OF
44/181 : 001496.OF
45/181 : 001193.OF
46/181 : 001319.OF
47/181 : 001396.OF
48/181 : 001476.OF
49/181 : 001404.OF
50/181 : 001313.OF
51/181 : 001416.OF
52/181 : 001409.OF
53/181 : 001245.OF
54

,ts_code,trade_date,fd_share,fund_type,market
0,001917.OF,2025-06-30,95950.0941,None,O
1,001917.OF,2025-03-31,97191.4172,None,O
2,001917.OF,2024-12-31,88017.2119,None,O
3,001917.OF,2024-09-30,79562.5216,None,O
4,001917.OF,2024-06-30,85613.0918,None,O
...,...,...,...,...,...
6879,040001.OF,2017-03-31,328795.0527,None,O
6880,040001.OF,2016-12-31,335648.5682,None,O
6881,040001.OF,2016-09-30,342470.2271,None,O
6882,040001.OF,2016-06-30,348888.1309,None,O


In [25]:
now_day = "2026-03-30"
def fill_fund_scale_time_series(df):
    """
    补全基金规模时间轴，按基金分组，空缺日期规模沿用最近一次有效值
    输入：df 包含 ts_code, trade_date, fd_share
    trade_date 格式：8位字符串，如 20200101
    """
    # 1. 复制数据，避免修改原表
    df = df.copy()
    
    # 2. 日期转 datetime 格式
    df['trade_date'] = pd.to_datetime(df['trade_date'], format='%Y%m%d')
    
    # 3. 按基金分组，为每个基金生成完整日期序列并补全
    def fill_group(group):
        # 按时间排序
        group = group.sort_values('trade_date')
        
        # if group['trade_date'].min() > pd.to_datetime(five_year_ago):
        #     print(f"基金 {group['ts_code'].iloc[0]} 的数据起始日期晚于 {five_year_ago}，可能无法补全到5年前")
        
        # 生成该基金的完整时间索引（从最早到最晚）
        date_range = pd.date_range(
            start=group['trade_date'].min(),
            end=now_day,
            freq='D'  # 按天补全
        )
        
        # 重新索引，补全日期
        group = group.set_index('trade_date').reindex(date_range)
        
        # 规模向下填充（空值用最近一次有效值）
        group['fd_share'] = group['fd_share'].fillna(method='ffill')
        
        # 基金代码向下填充
        group['ts_code'] = group['ts_code'].fillna(method='ffill')
        
        # 重置索引
        group = group.reset_index(names='trade_date')
        return group
    
    # 分组应用
    df_full = df.groupby('ts_code', group_keys=False).apply(fill_group)
    
    # 把日期转回 8 位字符串（保持你原来的格式）
    df_full['trade_date'] = df_full['trade_date'].dt.strftime('%Y%m%d')
    
    # 按基金+日期排序
    df_full = df_full.sort_values(['ts_code', 'trade_date']).reset_index(drop=True)
    
    return df_full
sdf_full = fill_fund_scale_time_series(sdf0)
sdf_full

/var/folders/ss/hq0_l7mj2f35ct8zlh27s1p80000gn/T/ipykernel_57051/399656299.py:33: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  group['fd_share'] = group['fd_share'].fillna(method='ffill')
/var/folders/ss/hq0_l7mj2f35ct8zlh27s1p80000gn/T/ipykernel_57051/399656299.py:36: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  group['ts_code'] = group['ts_code'].fillna(method='ffill')
/var/folders/ss/hq0_l7mj2f35ct8zlh27s1p80000gn/T/ipykernel_57051/399656299.py:33: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  group['fd_share'] = group['fd_share'].fillna(method='ffill')
/var/folders/ss/hq0_l7mj2f35ct8zlh27s1p80000gn/T/ipykernel_57051/399656299.py:36: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future ve

,trade_date,ts_code,fd_share,fund_type,market
0,20160331,000011.OF,18642.5278,None,O
1,20160401,000011.OF,18642.5278,NaN,NaN
2,20160402,000011.OF,18642.5278,NaN,NaN
3,20160403,000011.OF,18642.5278,NaN,NaN
4,20160404,000011.OF,18642.5278,NaN,NaN
...,...,...,...,...,...
661663,20260326,960000.OF,3922.2942,NaN,NaN
661664,20260327,960000.OF,3922.2942,NaN,NaN
661665,20260328,960000.OF,3922.2942,NaN,NaN
661666,20260329,960000.OF,3922.2942,NaN,NaN


In [26]:
sdf_full['ann_date'] = sdf_full['trade_date']
sdf = sdf_full[['ts_code', 'ann_date', 'fd_share']]
sdf['ann_date'] = pd.to_datetime(sdf['ann_date'], format="%Y%m%d")
sdf.set_index(['ann_date', 'ts_code'], inplace=True)
sdf.sort_index(inplace=True)
sdf

/var/folders/ss/hq0_l7mj2f35ct8zlh27s1p80000gn/T/ipykernel_57051/2625643176.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sdf['ann_date'] = pd.to_datetime(sdf['ann_date'], format="%Y%m%d")
/var/folders/ss/hq0_l7mj2f35ct8zlh27s1p80000gn/T/ipykernel_57051/2625643176.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sdf.sort_index(inplace=True)


fd_share
ann_date   ts_code               
2016-01-20 001877.OF   29144.8737
2016-01-21 001877.OF   29144.8737
2016-01-22 001877.OF   29144.8737
           002210.OF   20263.9215
2016-01-23 001877.OF   29144.8737
...                           ...
2026-03-30 540008.OF  155482.9886
           540009.OF   24682.4579
           540010.OF   16997.8038
           550008.OF  132721.6502
           960000.OF    3922.2942

[661668 rows x 1 columns]

In [27]:
sdf = sdf.loc[sdf.index.get_level_values("ann_date") >= ten_year_ago]
sdf

fd_share
ann_date   ts_code               
2016-03-30 001223.OF   24558.2148
           001645.OF   31816.0084
           001717.OF   22092.9795
           001718.OF   25035.9089
           001719.OF   30044.8207
...                           ...
2026-03-30 540008.OF  155482.9886
           540009.OF   24682.4579
           540010.OF   16997.8038
           550008.OF  132721.6502
           960000.OF    3922.2942

[661026 rows x 1 columns]

In [28]:
fsdf = ndf.merge(sdf, on=['ann_date', 'ts_code'], how='left')
fsdf['fd_share'] = fsdf.groupby('ts_code')['fd_share'].fillna(method='ffill')
fsdf['fd_share'] = fsdf.groupby('ts_code')['fd_share'].fillna(method='bfill')
fsdf

/var/folders/ss/hq0_l7mj2f35ct8zlh27s1p80000gn/T/ipykernel_57051/2375055054.py:2: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  fsdf['fd_share'] = fsdf.groupby('ts_code')['fd_share'].fillna(method='ffill')
/var/folders/ss/hq0_l7mj2f35ct8zlh27s1p80000gn/T/ipykernel_57051/2375055054.py:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  fsdf['fd_share'] = fsdf.groupby('ts_code')['fd_share'].fillna(method='ffill')
/var/folders/ss/hq0_l7mj2f35ct8zlh27s1p80000gn/T/ipykernel_57051/2375055054.py:3: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
 

nav_date  accum_nav    adj_nav     fd_share
ann_date   ts_code                                               
2016-03-30 000011.OF  20160329    13.6370  16.694700   18642.5278
           000309.OF  20160329     1.4450   1.445000   32296.7705
           000409.OF  20160329     1.3490   1.349000   36018.4498
           000471.OF  20160329     2.2190   2.219000  177241.4719
           000513.OF  20160329     1.3610   1.361000   74609.3099
...                        ...        ...        ...          ...
2026-03-31 540008.OF  20260330     2.9249   3.079717  155482.9886
           540009.OF  20260330     1.7303   1.714082   24682.4579
           540010.OF  20260330     4.1115   4.111500   16997.8038
           550008.OF  20260330     2.9682   3.523751  132721.6502
           960000.OF  20260330     2.2753   2.275300    3922.2942

[494930 rows x 4 columns]

In [29]:
# fsdf.to_feather('data/funds_with_scale.feather')
fsdf.to_feather('data/funds_with_scale_stock.feather')